# 🗄️ LangChain Vector Stores & FAISS Guide

Welcome to the **LangChain Vector Stores & FAISS** guide! This notebook demonstrates how to build an end-to-end vector search pipeline using **LangChain** and **FAISS** (Facebook AI Similarity Search).

---

### 🧠 Key Concepts & Objectives

Vector databases and vector stores are foundational components of **Retrieval-Augmented Generation (RAG)** systems. They allow you to store numerical vector representations (embeddings) of text chunks and perform ultra-fast semantic similarity search.

#### 🔄 Pipeline Workflow:
1. **Document Loading**: Ingest raw source data (e.g., plain text, PDFs, web pages).
2. **Text Chunking / Splitting**: Break documents into smaller, semantically coherent chunks with overlap.
3. **Embedding Generation**: Convert text chunks into high-dimensional vector embeddings via an embedding model (e.g., OpenAI `text-embedding-3-small`).
4. **Vector Store Indexing**: Store embeddings and associated document metadata in FAISS.
5. **Similarity Search**: Query the vector store using natural language to retrieve the most contextually relevant document chunks.

---

### 📦 Prerequisites
- `langchain`, `langchain-community`, `langchain-openai`, `langchain-text-splitters`
- `faiss-cpu` (or `faiss-gpu`)
- `python-dotenv`
- OpenAI API Key configured in your `.env` file (`OPENAI_API_KEY` or `OPEN_AI_KEY`)

## 🔑 Step 1: Environment Configuration

Load environment variables from the `.env` file and set the `OPENAI_API_KEY` for authenticating OpenAI embedding requests.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPEN_AI_KEY')

## 📦 Step 2: Import Dependencies

Import the necessary LangChain components:
- **`TextLoader`**: Ingests raw `.txt` documents into LangChain `Document` objects.
- **`RecursiveCharacterTextSplitter`**: Splits long text recursively by characters (paragraphs, sentences, words) while maintaining context.
- **`OpenAIEmbeddings`**: Generates high-dimensional vector embeddings using OpenAI's embedding models.
- **`FAISS`**: High-performance in-memory vector store for efficient similarity search and clustering of dense vectors.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 📄 Step 3: Document Loading & Text Splitting

1. **Load Document**: Use `TextLoader` to load `music_essay.txt`.
2. **Chunking**: Use `RecursiveCharacterTextSplitter` with:
   - `chunk_size=200`: Maximum number of characters per chunk.
   - `chunk_overlap=50`: Number of overlapping characters between consecutive chunks to ensure context is preserved across split boundaries.

In [3]:
loader = TextLoader("music_essay.txt")
text = loader.load()
text

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
doc = splitter.split_documents(text)
doc

[Document(metadata={'source': 'music_essay.txt'}, page_content='Western music theory is the study of the principles and structures that organize music. It helps musicians understand how notes, rhythms, melodies, harmonies, and forms work together to create a'),
 Document(metadata={'source': 'music_essay.txt'}, page_content='harmonies, and forms work together to create a musical composition. The foundation of Western music theory begins with the musical alphabet, which consists of seven basic notes: A, B, C, D, E, F, and'),
 Document(metadata={'source': 'music_essay.txt'}, page_content='of seven basic notes: A, B, C, D, E, F, and G. These notes repeat in higher or lower octaves.'),
 Document(metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
 Document(metadata={'source': 'music_essay.txt'

## 🧬 Step 4: Generate Embeddings & Build the FAISS Vector Store

1. **Initialize Embeddings**: Instantiate `OpenAIEmbeddings` using the lightweight and cost-effective `text-embedding-3-small` model.
2. **Vector Store Creation**: Use `FAISS.from_documents()` to:
   - Compute embedding vectors for all document chunks.
   - Index the vectors into an in-memory FAISS vector database.

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

db = FAISS.from_documents(doc, embedding=embeddings)
db

## 🔍 Step 5: Semantic Similarity Search

Query the FAISS vector database with a natural language question. FAISS embeds the query string and performs nearest-neighbor vector search to find and return the most relevant document chunks based on vector similarity.

In [5]:
query="how is minor scale different from major scale?"
docs=db.similarity_search(query=query)
docs

[Document(id='db2ecf8e-dff8-4d1c-9482-e78aad13bd2b', metadata={'source': 'music_essay.txt'}, page_content='pattern of whole and half steps and often sounds bright or cheerful. The natural minor scale has a different pattern and generally creates a darker or more emotional character. From scales, musicians'),
 Document(id='c9c8fc4d-860d-41a6-b0a4-00a5ef2c045d', metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
 Document(id='cd6033d7-c09d-4c2a-a225-02026b235a34', metadata={'source': 'music_essay.txt'}, page_content='more emotional character. From scales, musicians build chords, which are groups of three or more notes played together. Major, minor, diminished, and augmented chords are common types.'),
 Document(id='4c98e692-c675-40a2-a048-33469cd2f0cd', metadata={'source': 'music_essay.txt

In [6]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs

[Document(id='db2ecf8e-dff8-4d1c-9482-e78aad13bd2b', metadata={'source': 'music_essay.txt'}, page_content='pattern of whole and half steps and often sounds bright or cheerful. The natural minor scale has a different pattern and generally creates a darker or more emotional character. From scales, musicians'),
 Document(id='c9c8fc4d-860d-41a6-b0a4-00a5ef2c045d', metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
 Document(id='cd6033d7-c09d-4c2a-a225-02026b235a34', metadata={'source': 'music_essay.txt'}, page_content='more emotional character. From scales, musicians build chords, which are groups of three or more notes played together. Major, minor, diminished, and augmented chords are common types.'),
 Document(id='4c98e692-c675-40a2-a048-33469cd2f0cd', metadata={'source': 'music_essay.txt

In [7]:
docs = db.similarity_search_with_score(query)
docs

[(Document(id='db2ecf8e-dff8-4d1c-9482-e78aad13bd2b', metadata={'source': 'music_essay.txt'}, page_content='pattern of whole and half steps and often sounds bright or cheerful. The natural minor scale has a different pattern and generally creates a darker or more emotional character. From scales, musicians'),
  0.61521745),
 (Document(id='c9c8fc4d-860d-41a6-b0a4-00a5ef2c045d', metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
  0.8080884),
 (Document(id='cd6033d7-c09d-4c2a-a225-02026b235a34', metadata={'source': 'music_essay.txt'}, page_content='more emotional character. From scales, musicians build chords, which are groups of three or more notes played together. Major, minor, diminished, and augmented chords are common types.'),
  0.94762707),
 (Document(id='4c98e692-c675-40a2-a048-334

In [8]:
embedded_query = embeddings.embed_query(query)
docs = db.similarity_search_with_score_by_vector(embedded_query)
docs

[(Document(id='db2ecf8e-dff8-4d1c-9482-e78aad13bd2b', metadata={'source': 'music_essay.txt'}, page_content='pattern of whole and half steps and often sounds bright or cheerful. The natural minor scale has a different pattern and generally creates a darker or more emotional character. From scales, musicians'),
  0.61521745),
 (Document(id='c9c8fc4d-860d-41a6-b0a4-00a5ef2c045d', metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
  0.8080884),
 (Document(id='cd6033d7-c09d-4c2a-a225-02026b235a34', metadata={'source': 'music_essay.txt'}, page_content='more emotional character. From scales, musicians build chords, which are groups of three or more notes played together. Major, minor, diminished, and augmented chords are common types.'),
  0.94762707),
 (Document(id='4c98e692-c675-40a2-a048-334

In [9]:
#saving and loading
db.save_local("faiss_index")

In [10]:
new_db = db.load_local("faiss_index", embeddings=embeddings, allow_dangerous_deserialization=True)

In [11]:
new_db.similarity_search_with_score(query)

[(Document(id='db2ecf8e-dff8-4d1c-9482-e78aad13bd2b', metadata={'source': 'music_essay.txt'}, page_content='pattern of whole and half steps and often sounds bright or cheerful. The natural minor scale has a different pattern and generally creates a darker or more emotional character. From scales, musicians'),
  0.61521745),
 (Document(id='c9c8fc4d-860d-41a6-b0a4-00a5ef2c045d', metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
  0.8080884),
 (Document(id='cd6033d7-c09d-4c2a-a225-02026b235a34', metadata={'source': 'music_essay.txt'}, page_content='more emotional character. From scales, musicians build chords, which are groups of three or more notes played together. Major, minor, diminished, and augmented chords are common types.'),
  0.94762707),
 (Document(id='4c98e692-c675-40a2-a048-334

## ChromaDB

In [13]:
from langchain_chroma import Chroma

vector_db = Chroma.from_documents(documents=doc, embedding=embeddings)
vector_db

In [14]:
vector_db.similarity_search_with_score(query)

[(Document(metadata={'source': 'music_essay.txt'}, page_content='pattern of whole and half steps and often sounds bright or cheerful. The natural minor scale has a different pattern and generally creates a darker or more emotional character. From scales, musicians'),
  0.6152176856994629),
 (Document(metadata={'source': 'music_essay.txt'}, page_content='Scales are another important element. A scale is a sequence of notes arranged in ascending or descending order. The major scale follows a specific pattern of whole and half steps and often sounds'),
  0.8080886602401733),
 (Document(metadata={'source': 'music_essay.txt'}, page_content='more emotional character. From scales, musicians build chords, which are groups of three or more notes played together. Major, minor, diminished, and augmented chords are common types.'),
  0.9476274251937866),
 (Document(metadata={'source': 'music_essay.txt'}, page_content='of seven basic notes: A, B, C, D, E, F, and G. These notes repeat in higher or lo